# MiniCPM-o 4.5 — Full-Duplex Real-Time Video Commentary & Q&A

This notebook drives **MiniCPM-o 4.5** in **full-duplex (streaming) mode** so the model watches a video and listens to its audio one second at a time, and speaks back in real time — like a live commentator.

**Two modes:**
1. **Live commentary** — the model hears the original audio + sees the frames and narrates continuously.
2. **Interactive Q&A** — a pre-recorded question is injected into the audio stream at a chosen second; the model answers it, then keeps going.

> The output video contains **only the model's generated speech** — the original soundtrack is stripped from the render target, but is still fed to the model as *input* (via `audio_segments`).

## 0. Environment setup & dependencies

**One-time environment creation** — run these in a terminal, then pick the `minicpmo` kernel for this notebook (Kernel → Change Kernel):

```bash
conda create -n minicpmo python=3.10 -y
conda activate minicpmo
```

The cells below install the Python packages, `ffmpeg`, and the model weights **into the kernel you are running**. If you already have everything set up, skip to section 1.

> Launch Jupyter from this folder so the relative paths (`requirements.txt`, `assets/`) resolve. If an `import` fails right after installing, restart the kernel once and re-run.

In [ ]:
# Install the Python dependencies into the CURRENT kernel.
# `%pip` (not `!pip`) guarantees it targets this notebook's environment.
# NOTE: torch is pinned to 2.8.0. If you need a specific CUDA build, install
#       it first from https://pytorch.org, then run this cell.
%pip install -r requirements.txt

In [ ]:
# The helpers shell out to the system `ffmpeg` binary (to strip & mux audio),
# so it must be a real executable on PATH — not just the pip `imageio-ffmpeg` wheel.
import shutil

if shutil.which("ffmpeg") is None:
    print("ffmpeg not found on PATH — installing via conda...")
    %conda install -y -c conda-forge ffmpeg
    # Fallback on Debian/Ubuntu without conda:
    # !sudo apt-get update && sudo apt-get install -y ffmpeg
else:
    print(f"ffmpeg found: {shutil.which('ffmpeg')}")

### Download the MiniCPM-o 4.5 weights

Downloads the model from the Hugging Face Hub into your local cache and captures the snapshot folder as `LOCAL_PATH`. This runs **before** offline mode is switched on (the imports cell), so it needs network access the first time; every rerun just reuses the cache.

In [ ]:
import os
from huggingface_hub import snapshot_download

# Make sure we're allowed online for this one download (inference later runs offline).
os.environ["HF_HUB_OFFLINE"] = "0"
os.environ["TRANSFORMERS_OFFLINE"] = "0"

# Downloads once (large!), then just returns the local snapshot path on every rerun.
LOCAL_PATH = snapshot_download(repo_id="openbmb/MiniCPM-o-4_5")
print(f"Model weights available at:\n{LOCAL_PATH}")

## Before you run

Section 0 above installs everything into the current kernel. Beyond that you only need:

- A **CUDA-capable GPU** (the model runs in `bfloat16`).
- Your media under `assets/` (`football.mp4`, `audio1.wav`, etc).

Then run the cells top to bottom.

In [ ]:
import librosa
import torch
import numpy as np
import subprocess
import os
from minicpmo.utils import get_video_frame_audio_segments, generate_duplex_video
from transformers import AutoModel
import cv2

# Force everything offline so the model loads ONLY from the local snapshot
# (no network calls to huggingface.co at inference time).
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

## 1. Configuration

Paths and the injected-question settings. Point these at your own files.

In [ ]:
VIDEO_PATH     = "assets/vid.mp4"  # source video (frames + audio go INTO the model)
REF_AUDIO_PATH = "assets/audio1.wav"    # reference voice used by the TTS head
SAMPLE_RATE    = 16000

# Two injected questions
Q1_WAV    = "assets/q1.m4a"
Q1_AT_SEC = 5   # inject Q1 at second 5
Q1_TEXT   = "What did you just put on top of the cheese?"

Q2_WAV    = "assets/q2.m4a"
Q2_AT_SEC = 16  # inject Q2 at second 16
Q2_TEXT   = "List every ingredient that is added to the food processor in this video"

## 2. Helper — overlay the question caption

Reads the generated video frame by frame with OpenCV and burns the user's question onto the top of the frames during the seconds it was asked (Q&A mode only).

In [ ]:
def overlay_user_questions(video_in, video_out, q1_sec, q1_text, q2_sec, q2_text, duration_sec=4):
    """Reads the generated video frame-by-frame with OpenCV and hardcodes the
    user questions at the top of the video according to the target timestamps."""
    cap = cv2.VideoCapture(video_in)
    fps = cap.get(cv2.CAP_PROP_FPS)
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    temp_v = "temp_visuals.mp4"
    writer = cv2.VideoWriter(temp_v, fourcc, fps, (width, height))

    frame_idx = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        current_time = frame_idx / fps
        display_text = None

        if q1_sec <= current_time < (q1_sec + duration_sec):
            display_text = f"USER: {q1_text}"
        elif q2_sec <= current_time < (q2_sec + duration_sec):
            display_text = f"USER: {q2_text}"

        if display_text:
            font = cv2.FONT_HERSHEY_SIMPLEX
            scale = 0.9
            thick = 2
            t_size = cv2.getTextSize(display_text, font, scale, thick)[0]
            tx = int((width - t_size[0]) / 2)
            ty = 40

            cv2.rectangle(frame, (tx - 10, ty - t_size[1] - 10), (tx + t_size[0] + 10, ty + 10), (0, 0, 150), -1)
            cv2.putText(frame, display_text, (tx, ty), font, scale, (255, 255, 255), thick)

        writer.write(frame)
        frame_idx += 1

    cap.release()
    writer.release()

    cmd = ["ffmpeg", "-y", "-i", temp_v, "-i", video_in, "-map", "0:v", "-map", "1:a?", "-c:v", "libx264", "-c:a", "copy", video_out]
    subprocess.run(cmd, capture_output=True)

    if os.path.exists(temp_v): os.remove(temp_v)
    if os.path.exists(video_in): os.remove(video_in)
    print(f"Added top subtitles to -> {video_out}")

## 3. Helper — strip the source audio

Makes a **silent** copy of the source video. This is the target that the model's generated speech is muxed onto, so the original soundtrack never leaks into the output. The model still hears the original audio via `audio_segments`, extracted separately later.

In [ ]:
def strip_video_audio(input_path, output_path):
    """Create a silent copy of the source video (no audio track at all).
    generate_duplex_video will mux the model's generated speech onto this."""
    if not os.path.exists(input_path):
        raise FileNotFoundError(
            f"VIDEO_PATH does not exist on this machine: {input_path}. "
            f"Update VIDEO_PATH to the correct location of your video."
        )
    cmd = [
        "ffmpeg", "-y",
        "-i", input_path,
        "-an",
        "-c:v", "copy",
        output_path,
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print("---- ffmpeg stderr ----")
        print(result.stderr)
        print("------------------------")
        raise RuntimeError("ffmpeg failed to strip audio. See stderr above.")
    print(f"Created silent video (no original audio): {output_path}")

## 4. Helper — inject 2 questions into the audio stream

`_inject_question` overlays a single question's audio onto the chunk list starting at a given second. `build_audio_chunks_qa` calls it once per question (Q1, then Q2), falling back to the real narration audio (or silence) everywhere else.

In [ ]:
def _inject_question(chunks, q_path, q_sec, total):
    """Overlay a single question's audio onto the chunk list starting at q_sec,
    overwriting whatever real/background audio was there."""
    qaudio, _ = librosa.load(q_path, sr=SAMPLE_RATE, mono=True)
    q_chunk_count = int(np.ceil(len(qaudio) / SAMPLE_RATE))
    print(f"Question audio '{q_path}' is {len(qaudio)/SAMPLE_RATE:.1f}s -> spans chunks {q_sec} to {q_sec + q_chunk_count - 1}")

    for i in range(q_chunk_count):
        target_chunk = q_sec + i
        if target_chunk >= total:
            break
        offset = i * SAMPLE_RATE
        slice_ = qaudio[offset: offset + SAMPLE_RATE]
        if len(slice_) < SAMPLE_RATE:
            slice_ = np.pad(slice_, (0, SAMPLE_RATE - len(slice_)))
        chunks[target_chunk] = slice_.astype(np.float32)


def build_audio_chunks_qa(total, q1_path, q1_sec, q2_path=None, q2_sec=None, base_audio_segments=None):
    """Inject one or two questions into the per-second audio chunk stream.
    Elsewhere, fall back to the real video audio_segments if provided (so the
    model still hears the narrator between questions), otherwise silence."""
    if base_audio_segments is not None and len(base_audio_segments) >= total:
        chunks = [
            (base_audio_segments[i].copy() if base_audio_segments[i] is not None
             else np.zeros(SAMPLE_RATE, dtype=np.float32))
            for i in range(total)
        ]
    else:
        chunks = [np.zeros(SAMPLE_RATE, dtype=np.float32) for _ in range(total)]

    _inject_question(chunks, q1_path, q1_sec, total)

    if q2_path is not None and q2_sec is not None:
        _inject_question(chunks, q2_path, q2_sec, total)

    return chunks

## 5. The full-duplex streaming loop

This is the core. For every 1-second chunk we `streaming_prefill` the frame(s) + audio, then `streaming_generate` to pull any speech the model wants to emit. The loop mirrors the official HF reference: the **real** per-chunk audio waveform is fed in (not `None`), which is what lets the model actually hear the source. Finally, `generate_duplex_video` muxes the collected speech onto the silent video.

In [ ]:
def run_mode(model, ref_audio, video_frames, stacked_frames, audio_segments,
             mode_name, video_for_output, output_path, system_prompt,
             max_new_speak_tokens_per_chunk=20):
    """Iterate over chunks, passing the real per-chunk audio waveform
    (audio_segments[chunk_idx]) into streaming_prefill instead of None.
    This is what lets the model actually hear the source audio."""
    print(f"\n{'='*60}")
    print(f"Running: {mode_name}")
    print(f"{'='*60}\n")

    model_duplex = model.as_duplex()
    model_duplex.prepare(
        prefix_system_prompt=system_prompt,
        ref_audio=ref_audio,
        prompt_wav_path=REF_AUDIO_PATH,
    )

    results_log = []
    timed_output_audio = []

    for chunk_idx in range(len(audio_segments)+5):
        audio_chunk = audio_segments[chunk_idx] if chunk_idx < len(audio_segments) else None
        frame = video_frames[chunk_idx] if chunk_idx < len(video_frames) else None

        frame_list = []
        if frame is not None:
            frame_list.append(frame)
            if (stacked_frames is not None
                    and chunk_idx < len(stacked_frames)
                    and stacked_frames[chunk_idx] is not None):
                frame_list.append(stacked_frames[chunk_idx])

        model_duplex.streaming_prefill(
            audio_waveform=audio_chunk,
            frame_list=frame_list,
            max_slice_nums=1,
            batch_vision_feed=False,
        )

        result = model_duplex.streaming_generate(
            prompt_wav_path=REF_AUDIO_PATH,
            max_new_speak_tokens_per_chunk=max_new_speak_tokens_per_chunk,
        )

        if result["audio_waveform"] is not None:
            timed_output_audio.append((chunk_idx, result["audio_waveform"]))

        chunk_result = {
            "chunk_idx": chunk_idx,
            "is_listen": result["is_listen"],
            "text": result["text"],
            "end_of_turn": result["end_of_turn"],
            "current_time": result["current_time"],
            "audio_length": len(result["audio_waveform"]) if result["audio_waveform"] is not None else 0,
        }
        results_log.append(chunk_result)

        if result["is_listen"]:
            print(f"[{chunk_idx:03d}s] listening...")
        else:
            print(f"[{chunk_idx:03d}s] MODEL: {result['text']}")

    # generate_duplex_video writes subtitles.srt next to output_path but does not
    # create the parent directory itself, so make sure it exists first.
    output_dir = os.path.dirname(output_path)
    if output_dir:
        os.makedirs(output_dir, exist_ok=True)

    generate_duplex_video(
        video_path=video_for_output,
        output_video_path=output_path,
        results_log=results_log,
        timed_output_audio=timed_output_audio,
        output_sample_rate=24000,
    )
    print(f"\nSaved -> {output_path} (model-generated audio only, original audio removed)")

## 6. Load MiniCPM-o 4.5 (local, offline)

Loads the model from the local snapshot (downloaded in section 0) in `bfloat16` on the GPU and initializes the TTS head. `LOCAL_PATH` comes from the download cell above; override it manually in the next cell if you keep the weights elsewhere.

In [ ]:
# LOCAL_PATH was set by the download cell in section 0 (snapshot_download).
# If you keep the weights elsewhere / ran offline, set it manually instead, e.g.:
# LOCAL_PATH = "/home/user/.cache/huggingface/hub/models--openbmb--MiniCPM-o-4_5/snapshots/1f761131fa83f5ed3cd6f2f22b225c4501d154fa"

print(f"Loading model strictly from local path: {LOCAL_PATH}")
model = AutoModel.from_pretrained(
    LOCAL_PATH,              # direct path so it never calls huggingface.co
    trust_remote_code=True,
    local_files_only=True,   # redundant safety net
    attn_implementation="sdpa",
    torch_dtype=torch.bfloat16,
    init_vision=True,
    init_audio=True,
    init_tts=True,
)
model.eval().cuda()
model.init_tts()
ref_audio, _ = librosa.load(REF_AUDIO_PATH, sr=16000, mono=True)
print("Model ready.\n")

## 7. Extract frames + audio from the source video

Make a silent copy of the video (the mux target), then split the original into per-second frame + audio-waveform chunks. `audio_segments` is fed to the model as **input** so it can hear the source.

In [ ]:
import os
os.makedirs("outputs", exist_ok=True)

silent_video_path = "input_silent.mp4"
strip_video_audio(VIDEO_PATH, silent_video_path)

video_frames, audio_segments, stacked_frames = get_video_frame_audio_segments(
    VIDEO_PATH, stack_frames=1, use_ffmpeg=True, adjust_audio_length=True
)
# set use_ffmpeg = False if video has no audio
total_chunks = len(video_frames)

if audio_segments is None:
    # VIDEO_PATH has no audio track, so get_video_frame_audio_segments degrades to
    # vision-only and returns None here. run_mode indexes len(audio_segments)
    # unconditionally, so backfill silence rather than leaving it None.
    print(f"{VIDEO_PATH} has no audio track -- feeding silence instead of source audio.\n")
    audio_segments = [np.zeros(SAMPLE_RATE, dtype=np.float32) for _ in range(total_chunks)]

print(f"Total chunks: {total_chunks} ({total_chunks}s of video)\n")

## 8. Mode 1 — Live commentary [DISABLED]

The model receives the frames **and** the original audio and narrates continuously. Writes `new_outputs/rht.mp4` containing only the model's spoken commentary.

This mode is commented out below — only Mode 2 (Q&A) is active. Uncomment the cell to re-enable it.

In [ ]:
run_mode(
    model, ref_audio, video_frames, stacked_frames, audio_segments,
    mode_name="Live Narration",
    video_for_output=silent_video_path,
    output_path="audio1.mp4",
    system_prompt="""You are a proactive, real-time commentator. Continuously describe the scene as it happens. """,
    max_new_speak_tokens_per_chunk=30,
)


## 9. Mode 2 — Q&A with 2 injected questions [ACTIVE]

Injects the pre-recorded questions at `Q1_AT_SEC` and `Q2_AT_SEC`, lets the model answer each mid-stream (staying silent otherwise), then burns both question captions on top. Writes `output_cooking_qa.mp4`.

In [ ]:
qa_chunks = build_audio_chunks_qa(
    total_chunks,
    Q1_WAV, Q1_AT_SEC,
    q2_path=Q2_WAV, q2_sec=Q2_AT_SEC,
    base_audio_segments=audio_segments,
)

temp_qa_out = "raw_output_cooking_qa.mp4"

run_mode(
    model, ref_audio, video_frames, stacked_frames, qa_chunks,
    mode_name="Cooking Narration + 2 Injected Questions",
    video_for_output=silent_video_path,
    output_path=temp_qa_out,
    system_prompt=(
        """You are watching a cooking video. Stay silent and do not narrate or describe
anything on your own.
Only speak when you are directly asked a question.
When asked, answer using only what is clearly shown in the video,
in as few words as possible — no extra commentary, no restating the question.
If unsure, say "not clear" instead of guessing."""
    ),
    max_new_speak_tokens_per_chunk=20,
)

overlay_user_questions(
    video_in=temp_qa_out,
    video_out="output_cooking_qa.mp4",
    q1_sec=Q1_AT_SEC,
    q1_text=Q1_TEXT,
    q2_sec=Q2_AT_SEC,
    q2_text=Q2_TEXT,
    duration_sec=4,
)